# 05.07.03 Multi-Turn Conversation and Follow-Up Generation

**Phase 05 — Track B: LLM-Powered Patient Interview Assistant**
**Primary Model:** `Qwen3-4B-Instruct-2507`

### Research Objectives
1. Design multi-turn interview state management.
2. Generate context-aware follow-up questions for incomplete patient answers.
3. Support MSW acceptance, editing, or rejection of suggested follow-ups.


In [1]:
from typing import List, Dict

def build_followup_context(history: List[Dict[str, str]], latest_utterance: str) -> str:
    """Formats conversation history for Qwen3-4B follow-up generation."""
    formatted = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history])
    return f"{formatted}\nPATIENT: {latest_utterance}".strip()


In [ ]:
# --- CHAPTER 4 VISUALIZATION: Interview Response Token Length & Missing Slot Follow-Up Triggers (N=1,044) ---
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../../..').resolve()
PROJECT_ROOT = _find_project_root(Path.cwd())
VIZ_DIR = PROJECT_ROOT / 'reports' / 'phase_05' / 'visualizations' / 'llm_interview_assistant'
VIZ_DIR.mkdir(parents=True, exist_ok=True)
df_clean = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'ofphil_cleaned_tabular.csv')
word_counts = df_clean['interview_responses'].dropna().astype(str).str.split().str.len()
missing_slots = {
    'Monthly Income Missing': int(df_clean['monthly_family_income'].isna().sum()),
    'Obligations Missing': int(df_clean['monthly_household_obligations_expenses'].isna().sum()),
    'Dependents Missing': int(df_clean['number_of_dependents'].isna().sum()),
    'Education Missing': int(df_clean['educational_attainment'].isna().sum()),
    'Housing Condition Missing': int(df_clean['housing_condition'].isna().sum()),
}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))
ax1.hist(word_counts, bins=20, color='#386cb0', edgecolor='black', alpha=0.85)
ax1.set_title(f'Patient Interview Response Length Distribution (n={len(word_counts)})', fontsize=11, fontweight='bold')
ax1.set_xlabel('Word Count per Interview Transcript', fontsize=10, fontweight='bold')
ax1.set_ylabel('Patient Count', fontsize=10, fontweight='bold')
ax2.barh(list(missing_slots.keys())[::-1], list(missing_slots.values())[::-1], color='#d95f02', edgecolor='black')
ax2.set_title('Intake Missing-Slot Triggers Requiring Qwen3 Follow-Up (N=1,044)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Cases Triggering Follow-Up Clarification', fontsize=10, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'interview_followup_triggers.png', dpi=300, bbox_inches='tight')
fig.savefig(VIZ_DIR / 'interview_followup_triggers.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)
